Mount Google Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

base_dir = '/content/drive/MyDrive/laby'  # Adjust if needed
excel_path = os.path.join(base_dir, 'Data-Laby 5-12.xlsx')
image_dir = os.path.join(base_dir, 'images')

df = pd.read_excel(excel_path)
df['filename'] = [f"{i}.jpg" for i in range(1, 21)]
df['filepath'] = df['filename'].apply(lambda x: os.path.join(image_dir, x))
df = df[['filepath', 'LC', 'MD', 'DP', 'TT']]
train_df, test_df = train_test_split(df, test_size=5, random_state=42)


Preprocess the Images (RGB for CNN/Augmentation)

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

IMG_SIZE = 224

def preprocess_image_rgb(image_path):
    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))  # RGB
    img = img_to_array(img) / 255.0
    return img

X_train_rgb = np.array([preprocess_image_rgb(p) for p in train_df['filepath']])
X_test_rgb = np.array([preprocess_image_rgb(p) for p in test_df['filepath']])

scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(train_df[['LC', 'MD']].values)
y_test_scaled = scaler.transform(test_df[['LC', 'MD']].values)


# **Data Augmentation + CNN Training**

 Step 1: Import and Configure ImageDataGenerator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

datagen.fit(X_train_rgb)


Step 2: Define a CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

aug_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='linear')  # Output: LC and MD
])

aug_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
aug_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,650 (91.20 MB)

 Trainable params: 23,907,650 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

Step 3: Train the Model with Augmented Data

In [ ]:
batch_size = 4
epochs = 50

history_aug = aug_model.fit(
    datagen.flow(X_train_rgb, y_train_scaled, batch_size=batch_size),
    epochs=epochs,
    validation_data=(X_test_rgb, y_test_scaled),
    steps_per_epoch=len(X_train_rgb) // batch_size
)


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 957ms/step - loss: 499.0337 - mae: 13.9007 - val_loss: 128.9079 - val_mae: 11.3466
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - loss: 158.5400 - mae: 12.1419 - val_loss: 90.1936 - val_mae: 9.4912
Epoch 3/50


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 963ms/step - loss: 48.6591 - mae: 6.2668 - val_loss: 2.4064 - val_mae: 1.3325
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 157ms/step - loss: 1.5798 - mae: 1.0793 - val_loss: 4.3328 - val_mae: 1.7780
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 873ms/step - loss: 10.6740 - mae: 2.7133 - val_loss: 0.1052 - val_mae: 0.2422
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - loss: 1.3194 - mae: 0.9286 - val_loss: 0.2301 - val_mae: 0.3772
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 838ms/step - loss: 1.1267 - mae: 0.8565 - val_loss: 0.1614 - val_mae: 0.3339
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - loss: 0.2743 - mae: 0.4307 - val_loss: 0.1645 - val_mae: 0.3319
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - loss: 0.4319 - mae: 0.5413 - val_loss: 0.1721 - val_mae: 0.3384
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - loss: 0.2343 - mae: 0.4458 - val_loss: 0.1570 - val_mae: 0.3251
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 861ms/step - loss: 0.1388 - mae: 0.2971 - val

Step 4: Predict and Evaluate

In [ ]:
y_pred_scaled = aug_model.predict(X_test_rgb)
y_pred = scaler.inverse_transform(y_pred_scaled)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

true_lc = scaler.inverse_transform(y_test_scaled)[:, 0]
true_md = scaler.inverse_transform(y_test_scaled)[:, 1]
pred_lc = y_pred[:, 0]
pred_md = y_pred[:, 1]

print("📏 LC Evaluation (Augmented CNN):")
print("MAE:", mean_absolute_error(true_lc, np.round(pred_lc)))
print("R²:", r2_score(true_lc, np.round(pred_lc)))

print("\n📏 MD Evaluation (Augmented CNN):")
print("MAE:", mean_absolute_error(true_md, pred_md))
print("R²:", r2_score(true_md, pred_md))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
📏 LC Evaluation (Augmented CNN):
MAE: 3.0
R²: -0.23831775700934554

📏 MD Evaluation (Augmented CNN):
MAE: 1.3582866191864014
R²: -0.5196810676670935


In [ ]:
from sklearn.metrics import mean_squared_error

print("📏 LC Evaluation (Augmented CNN):")
print("MAE:", mean_absolute_error(true_lc, np.round(pred_lc)))
print("MSE:", mean_squared_error(true_lc, np.round(pred_lc)))
print("R²:", r2_score(true_lc, np.round(pred_lc)))

print("\n📏 MD Evaluation (Augmented CNN):")
print("MAE:", mean_absolute_error(true_md, pred_md))
print("MSE:", mean_squared_error(true_md, pred_md))
print("R²:", r2_score(true_md, pred_md))


📏 LC Evaluation (Augmented CNN):
MAE: 3.0
MSE: 10.599999999999998
R²: -0.23831775700934554

📏 MD Evaluation (Augmented CNN):
MAE: 1.3582866191864014
MSE: 3.2825111061609222
R²: -0.5196810676670935
